# WaterSoftHack 2026 — Day 3: Hybrid Pipeline (Hands-On)

**The finale.** Nothing in this notebook is new. Tuesday you trained a cloud now-cast; Wednesday you
programmed a smart gauge. Today you wire them into one early-warning system and watch it beat both
halves working alone.

The system, end to end:

1. **A fleet** of five smart gauges strung down a basin, with a storm rolling through them in sequence.
2. **A cloud brain** that ingests their messages and keeps a live basin picture.
3. **Trigger & burst:** the upstream gauge's alert makes the cloud forecast *every downstream reach* at once.
4. **The loop closes:** the cloud pushes **storm mode** back down, and we measure the extra warning time it buys.

> Same JupyterHub, same login. Run each cell with **Shift+Enter**. The slogan of the day:
> **let each layer do what only it can do.**

---
## Part 0 — The basin and the storm

Five gauges down one river, from headwaters (gauge 0) to the town we are protecting (gauge 4). A flash
flood begins upstream and travels downstream, hitting each gauge about **90 minutes** after the one above
it. That travel time is the whole opportunity: if gauge 0 detects the rise, the town at gauge 4 has hours.

**That downstream lag is the prize.** A purely local warning at the town (gauge 4) fires when the
water is already there. A *hybrid* system uses gauge 0's alarm plus a cloud forecast to warn the town
while its river is still quiet. Let's build it.

---
## Part 1 — The fleet (Wednesday's gauge, x5)

Here is the smart gauge from yesterday, trimmed to the essentials: adaptive sampling, the two detection
rules, hourly summaries, and store-and-forward. The one addition for today is a **radio**: instead of a
private log, each gauge `publish`es messages to a shared **bus** the cloud can read - and it `subscribe`s
to a `config` topic the cloud can write. That is the two-way street from the lecture, in code.

---
## Part 2 — The cloud brain

The cloud subscribes to the bus. Its job is **fusion**: keep a live picture of every gauge, notice
alarms the instant they arrive, and - crucially - decide what to do about them. For now, build the
ingest-and-track half. It reads every message published so far and reports the basin state.

---
## Part 3 — Run the baseline: hybrid detection, healthy network

Run the fleet through the whole storm with the network up, feeding the cloud as messages arrive. No
storm mode yet - just filter-and-forward plus the cloud watching. We will see the alerts march
downstream gauge by gauge.

**Watch the cascade.** The alert fires at gauge 0 first, then gauge 1 about 90 minutes later, and so
on down to the town. And look at the basin board: partway through the storm the cloud sees the upstream
gauges in FLOOD while the town is still calm. Each gauge, on its own, only knows its own water. The
cloud is the first entity that can see the *pattern* - an alarm marching downstream - and act on it
before the wave reaches gauge 4.

---
## Part 4 — Trigger & burst

The moment the cloud sees gauge 0's alert, it does what no microcontroller can: using the **basin
topology** - which reach flows into which, and how fast - it projects when the flood will reach *every
downstream gauge*. A single reach cannot do this; it has no idea what is upstream. This is flood
routing, and the cloud bursts it across cores the instant the alarm lands. One 60-byte message from a
$5 chip spins up the supercomputer.

The cloud now holds something no single gauge could produce: a **basin-wide flood-arrival forecast**,
generated the instant the first alarm arrived. The town at gauge 4 learns, at 07:00, that its river will
flood in the afternoon - hours before its own water has moved a foot. That is trigger-and-burst: idle
cheaply, react massively on the event, release. The cost is pennies of credits, because the cores are
held for seconds rather than left running between storms.

---
## Part 5 — Close the loop: push storm mode down

Now the downstream direction. The cloud has seen gauge 0 alarm, so it *knows* the wave is coming for
gauges 1 to 4. It reaches down and puts those gauges into **storm mode**: sample every minute and drop
the rise threshold to a hair trigger. We rerun the fleet, but this time the cloud re-tunes each
downstream gauge the moment the wave upstream trips - and we measure the extra warning time.

**Read the last column.** Gauge 0 gains nothing - it is the first to see the storm, so there is
nothing upstream to warn it. But every downstream gauge alarms **earlier** because the cloud, seeing the
alert march down the basin, lowered their triggers before their own water rose. The town at gauge 4 gets
the biggest head start. That is the hybrid payoff: **foresight the edge cannot have, delivered as
configuration the edge can act on.**

---
## Part 6 — The scoreboard: three architectures, one storm

Finally, the whole week on one table. We score the same flood three ways: cloud-only (Tuesday's dumb
pipe), edge-only (Wednesday's lonely hero), and today's hybrid. The metric that matters is **warning
time at the town** - and whether the warning survives an outage.

**The hybrid row wins the one number that matters** - lead time at the town - *and* keeps the
outage-survival that the edge gave us yesterday. Cloud-only has foresight but dies in the outage;
edge-only survives but has no foresight for the town; hybrid has both, because each layer did what only
it could do.

---
## Wrap-up

You assembled the whole early-warning system:

- **a fleet** of smart gauges publishing to a shared bus,
- **a cloud brain** that fused their messages into a live basin picture,
- **trigger & burst** - one edge alert summoning a basin-wide forecast in seconds,
- **a closed loop** - storm mode pushed down, buying the town extra warning time,
- and a **scoreboard** proving the hybrid beats either half alone.

### Where this goes

Every piece here is modular and yours to fork. Swap our design storm for live USGS data; add a gateway
tier; replace the rule-based detector with a TinyML model trained in the cloud and shrunk for the edge.
The architecture you just built is the same one behind IFIS and the National Water Model - you have the
skills now. Go point them at your own Week 2 question.

---
## Optional challenges (if you finish early)

1. **Longer basin.** Set `N_GAUGES = 10`. Does the town's lead time grow? Why?
2. **Cut the backhaul.** Give gauge 2 a 6-hour outage during the storm (`network=`). Does the hybrid
   still warn the town? Trace which layer carried it.
3. **Tune storm mode.** Change the pushed-down `rise_alert` from 0.4 to 0.2. More lead time, or more
   false alarms on the daily wiggle? Where is the sweet spot?
4. **Real rivers.** Replace `storm_at` with live USGS stage for five gauges down an actual Iowa river
   (use Day 2's `fetch_live_stage`). Does the cascade still show up?
5. **A gateway tier.** Add a class that sits between three gauges and the cloud, aggregating their
   messages into one uplink. How many bus messages does it save?